# Blackboard DEMO

## Setup

In [1]:
from src.board import Board

question = """
Целеполагание г. Гатчина (Ленинградская область) на 2026-2035 годы
"""

board = Board(question=question)

In [2]:
from src.agents.factories import *

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [3]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(str.join(', ', expert.info.values()))

728347, Градостроительный планировщик (урбанист), Эксперт в области долгосрочного развития городской инфраструктуры, земельного планирования, транспортных систем и пространственного размещён
3a4021, Экономист регионального развития, Специалист по анализу экономических тенденций, инвестиционных проектов, бюджетного планирования и формированию конкурентоспособных отраслей
6ef77d, Стратегический консультант по муниципальному управлению, Профессионал в разработке и реализации публичных стратегий, управлении проектами, взаимодействии с гражданским обществом и оценке социально‑


In [4]:
from src.searcher import WikipediaSearcher
wikipedia_searcher = WikipediaSearcher()

planner_agent = create_planner_agent(board)
wikipedia_agent = create_wikipedia_agent(wikipedia_searcher, board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)
decider_agent = create_decider_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent] # wikipedia_agent, 
controller_agent = create_controller_agent(role_agents, board)

In [5]:
for agent in [*role_agents, controller_agent]:
    response_format = agent.response_format
    if response_format is not None:
        agent.checkpointer.allowed_msgpack_modules = [
            (response_format.__module__, response_format.__name__),
        ]

## Experiment

In [6]:
from src.agents import Agent

def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

In [7]:
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage

is_final = False

for i in range(3):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [SystemMessage(f'Записи на доске:\n{str(board.notes)}')], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join(', ', [a.role.name for a in agents]))

    for a in tqdm(agents):
        try:
            response = a.invoke([SystemMessage(f'Записи на доске:\n{str(board.notes)}')], force=False)
        except:
            print(a.role.name + ' перегрелся')
            continue

        if a == decider_agent: # DECIDER AGENT
            note = response.note
            is_final = response.is_final
        elif a == cleaner_agent: # CLEANER AGENT
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
            continue
        else: # OTHER AGENTS
            note = response
        
        note_id = board.add_note(note, a.id, a.role.name)

        if is_final:
            break

    if is_final:
        break

Итерация 1
Планировщик, Градостроительный планировщик (урбанист), Экономист регионального развития, Стратегический консультант по муниципальному управлению, Критик, Уборщик, Арбитр


 14%|█▍        | 1/7 [00:01<00:10,  1.78s/it]

Планировщик перегрелся


 29%|██▊       | 2/7 [00:04<00:12,  2.49s/it]

Градостроительный планировщик (урбанист) перегрелся


...

Арбитр перегрелся
Итерация 2


Планировщик, Градостроительный планировщик (урбанист), Экономист регионального развития, Стратегический консультант по муниципальному управлению, Критик, Уборщик, Арбитр


100%|██████████| 7/7 [01:33<00:00, 13.39s/it]


Итерация 3
Критик, Уборщик


100%|██████████| 2/2 [00:35<00:00, 17.79s/it]


In [8]:
board.notes

[Note(content='План решения задачи:\n\n1. **Формулировка задачи** – собрать и систематизировать информацию о текущем состоянии города Гатчина, определить ключевые направления развития и сформировать цели на 2026‑2035 годы.\n\n2. **Сбор исходных данных**\n   - демографические, экономические, социальные и инфраструктурные показатели;\n   - существующие стратегии и программы развития (на уровне муниципалитета, региона, федеральные проекты);\n   - результаты предыдущих планов (2016‑2025) и их оценка.\n\n3. **Анализ заинтересованных сторон**\n   - органы местного самоуправления, бизнес‑сообщество, общественные организации, жители.\n   - проведение интервью/опросов, формирование требований и ожиданий.\n\n4. **SWOT‑анализ** города (сильные/слабые стороны, возможности, угрозы).\n\n5. **Определение стратегических направлений** (например, экономическое развитие, жилищно‑коммунальное хозяйство, транспорт, экология, культура, цифровизация).\n\n6. **Формулирование целей и задач** для каждого направ

In [9]:
board.print()

╭──────────────────────────────────────── 📌 Планировщик  ─────────────────────────────────────────╮
│ План решения задачи по целеполаганию Гатчины на 2026‑2035 годы: формулировка задачи, сбор        │
│ данных, анализ стейкхолдеров, SWOT, определение направлений, формулирование SMART‑целей,         │
│ разработка мероприятий, оценка KPI, подготовка документа, публичное обсуждение, план реализации. │
│ След                                                                                             │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ План решения задачи:                                                                             │
│                                                                                                  │
│ 1. **Формулировка задачи** – собрать и систематизировать информацию о текущем состоянии города   │
│ Гатчина, определить ключевые направления развития и сформировать цели на 2026‑2035 годы.         │
│                                                                                                  │
│ 2. **Сбор исходных данных**                                                                      │
│    - демографические, экономические, социальные и инфраструктурные показатели;                   │
│    - существующие стратегии и программы развития (на уровне муниципалитета, региона, федеральные │
│ проекты);                                                                                        │
│    - результаты предыдущих планов (2016‑2025) и их оценка.                                       │
│                                                                                                  │
│ 3. **Анализ заинтересованных сторон**                                                            │
│    - органы местного самоуправления, бизнес‑сообщество, общественные организации, жители.        │
│    - проведение интервью/опросов, формирование требований и ожиданий.                            │
│                                                                                                  │
│ 4. **SWOT‑анализ** города (сильные/слабые стороны, возможности, угрозы).                         │
│                                                                                                  │
│ 5. **Определение стратегических направлений** (например, экономическое развитие,                 │
│ жилищно‑коммунальное хозяйство, транспорт, экология, культура, цифровизация).                    │
│                                                                                                  │
│ 6. **Формулирование целей и задач** для каждого направления на 2026‑2035 годы (SMART‑цели).      │
│                                                                                                  │
│ 7. **Разработка мероприятий и проектов** – конкретные инициативы, сроки, ответственные, ресурсы. │
│                                                                                                  │
│ 8. **Оценка эффективности** – показатели KPI, методика мониторинга и контроля.                   │
│                                                                                                  │
│ 9. **Подготовка итогового документа** – текст стратегического плана, визуализации, резюме для    │
│ публичного обсуждения.                                                                           │
│                                                                                                  │
│ 10. **Публичное обсуждение и согласование** – сбор обратной связи, корректировка, утверждение.   │
│                                                                                                  │
│ 11. **План реализации** – календарный график, бюджет, распре

╭────────────────────── 📌 Градостроительный планировщик (урбанист) [728347] ──────────────────────╮
│ Предложение по дальнейшему развитию задачи целеполагания Гатчины: расширенный перечень исходных  │
│ данных, список стейкхолдеров, предложения по стратегическим направлениям, KPI и этапам           │
│ реализации.                                                                                      │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ ### Предложение по дальнейшему развитию задачи «Целеполагание г. Гатчина (Ленинградская область) │
│ на 2026‑2035 годы»                                                                               │
│                                                                                                  │
│ #### 1. Расширенный перечень исходных данных                                                     │
│ | Категория | Источник | Период | Комментарий |                                                  │
│ |-----------|----------|--------|-------------|                                                  │
│ | Демография | Росстат, муниципальная статистика | 2010‑2025 | Состав населения, миграционные    │
│ потоки, возрастные группы, уровень рождаемости/смертности. |                                     │
│ | Трудовой рынок | Портал «Работа в России», региональная служба занятости | 2015‑2025 | Уровень │
│ безработицы, структура занятости, востребованные профессии, наличие квалифицированных кадров. |  │
│ | Жилищный фонд | Департамент ЖКХ, кадастровая служба | 2010‑2025 | Площадь жилого фонда,        │
│ состояние зданий, доля коммунального и частного жилья, доступность жилья для разных доходных     │
│ групп. |                                                                                         │
│ | Транспорт | ДТЭК, «Гатчинский транспорт», GPS‑данные общественного транспорта | 2015‑2025 |    │
│ Интенсивность движения, загруженность дорог, доступность ОТК, количество велосипедных дорожек,   │
│ уровень паркингов. |                                                                             │
│ | Экология | Роспотребнадзор, региональная служба охраны окружающей среды | 2015‑2025 | Уровень  │
│ загрязнения воздуха и воды, количество зелёных насаждений, состояние рек и озёр, количество      │
│ «тёплых» зон. |                                                                                  │
│ | Цифровая инфраструктура | Департамент ИТ, оператор связи | 2018‑2025 | Доступность             │
│ широкополосного интернета, уровень цифровой грамотности, количество «умных» сервисов в городе. | │
│ | Экономика | Портал «Бизнес‑инфо», региональная инвестиционная служба | 2015‑2025 | ВВП         │
│ муниципального уровня, отраслевой состав, инвестиционные проекты, уровень предпринимательской    │
│ активности. |                                                                                    │
│ | Социальные услуги | Департамент соцзащиты, школы, больницы | 2015‑2025 | Доступность и         │
│ качество образования, медицины, культуры, спорта. |                                              │
│                                                                                                  │
│ #### 2. Расширенный список заинтересованных сторон (стейкхолдеров)                               │
│ | Группа | Представители | Интересы/Ожидания |                                                   │
│ |--------|----------------|-------------------|                                                  │
│ | Муниципальная власть | Глава администрации, Совет депутатов | Эффективное управление,          │
│ привлечение влож‑                                                                                │
│                                                             

╭────────────────────────── 📌 Экономист регионального развития [3a4021] ──────────────────────────╮
│ Экономический план для целеполагания Гатчины 2026‑2035: набор необходимых экономических данных,  │
│ стратегические направления, цели, мероприятия и ожидаемый эффект.                                │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ ### Экономический вклад в целеполагание г. Гатчина (2026‑2035) – заметка экономиста              │
│ регионального развития (id = 3a4021)                                                             │
│                                                                                                  │
│ #### 1. Ключевые экономические показатели, которые необходимо собрать и проанализировать         │
│ | Показатель | Источник | Период | Примечание |                                                  │
│ |------------|----------|--------|-----------|                                                   │
│ | **Валовый региональный продукт (ВРП) муниципального уровня** | Росстат, региональная служба    │
│ экономического развития | 2010‑2025 | Динамика роста, структура по отраслям, доля ВРП в ВРП      │
│ Ленобласти |                                                                                     │
│ | **Отраслевая структура экономики** | Портал «Бизнес‑инфо», региональная инвестиционная служба  │
│ | 2015‑2025 | Доля промышленности, строительства, услуг, туризма, ИТ‑сектора |                   │
│ | **Инвестиционный климат** | Инвестиционный портал Ленобласти, местный инвестиционный фонд |    │
│ 2015‑2025 | Объём привлечённых инвестиций (прямые, портфельные), количество новых проектов,      │
│ средний срок окупаемости |                                                                       │
│ | **Трудовой рынок** | Портал «Работа в России», служба занятости | 2015‑2025 | Уровень          │
│ безработицы, доля занятых в приоритетных отраслях, наличие квалифицированных кадров,             │
│ миграционные потоки (приток/отток специалистов) |                                                │
│ | **Налоговая база** | Департамент финансов, ФНС | 2015‑2025 | Налоговые поступления (НДФЛ, УСН, │
│ налог на прибыль), доля налогов от малого и среднего бизнеса |                                   │
│ | **Экспорт/импорт товаров и услуг** | Таможенная служба, региональная торговая палата |         │
│ 2015‑2025 | Объём внешнеторговой деятельности, ключевые экспортные товары (например,             │
│ деревообработка, легкая промышленность) |                                                        │
│ | **Стоимость земли и недвижимости** | Кадастровая служба, агентства недвижимости | 2015‑2025 |  │
│ Средняя цена за м², динамика цен, доступность для инвесторов и населения |                       │
│ | **Эффективность муниципального бюджета** | Департамент финансов | 2015‑2025 | Доля расходов на │
│ капитальные вложения, доля расходов на текущие нужды, уровень дефицита/профицита |               │
│                                                                                                  │
│ #### 2. Стратегические экономические направления (2026‑2035)                                     │
│ | Направление | Цель (SMART) | Ключевые мероприятия | Ожидаемый экономический эффект |           │
│ |---|---                                                                                         │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ экономика, Гатчина, 2026‑2035, инвестиции, трудовой рынок   

╭────────────── 📌 Стратегический консультант по муниципальному управлению [6ef77d] ───────────────╮
│ Стратегический план‑дорожная карта для целеполагания г. Гатчина (2026‑2035) от Стратегического   │
│ консультанта: управленческая структура, интегрированная база данных, процесс формирования целей, │
│ набор SMART‑целей, приоритетные направления, система KPI, механизм публичного контроля,          │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ ### Стратегический план‑дорожная карта для целеполагания г. Гатчина (2026‑2035) – вклад          │
│ Стратегического консультанта по муниципальному управлению (id = 6ef77d)                          │
│                                                                                                  │
│ #### 1. Управленческая структура проекта                                                         │
│ | Уровень | Ответственный | Функции |                                                            │
│ |---------|----------------|---------|                                                           │
│ | **Координационный совет** | Глава администрации + Председатель Совета депутатов | Утверждение  │
│ стратегии, контроль бюджета, согласование межведомственных вопросов |                            │
│ | **Рабочая группа «Стратегия 2026‑2035»** | Руководитель проекта (мэр/зам. мэра) +              │
│ представители 5‑ти тематических подгрупп (экономика, инфраструктура, соцсфера, экология,         │
│ цифровизация) | Разработка целей, формулирование мероприятий, подготовка KPI |                   │
│ | **Подгруппы** | Специалисты профильных департаментов + приглашённые эксперты + представители   │
│ бизнеса и НКО | Сбор и анализ данных, подготовка предложений, проведение экспертизы |            │
│ | **Экспертный совет граждан** | Представители жителей (жильё, молодёжь, пенсионеры),            │
│ общественные организации, бизнес‑ассоциации | Публичные обсуждения, сбор обратной связи,         │
│ мониторинг реализации |                                                                          │
│                                                                                                  │
│ #### 2. Интегрированная база данных (Data‑hub)                                                   │
│ - **Техническая платформа** – облачное хранилище с открытым API, совместимое с системами         │
│ «ГИС‑Гатчина», «Электронный бюджет», «Портал открытых данных».                                   │
│ - **Ключевые наборы данных** (объединяют сведения из записей 1‑3):                               │
│   1. Демография и миграция (Росстат, муниципальная статистика);                                  │
│   2. Трудовой рынок и квалификационный потенциал (служба занятости, образовательные учреждения); │
│   3. Жилищный фонд и инфраструктура (Департамент ЖКХ, кадастр);                                  │
│   4. Транспорт и мобильность (ДТЭК, GPS‑данные);                                                 │
│   5. Экология и климат (Роспотребнадзор, региональная служба ООС);                               │
│   6. Цифровая инфраструктура (Департамент ИТ, операторы связи);                                  │
│   7. Экономика и инвестиционный климат (Бизнес‑инфо, инвестиционный фонд);                       │
│   8. Социальные услуги (образование, медицина, культура, спорт).                                 │
│ - **Обновление** – ежеквартальное автоматическое импортирование из официальных источников +      │
│ ежегодный «Год‑от                                                                                │
│                                                                                                  │
│ ---                                                         

╭─────────────────────────────────────────── 📌 Арбитр  ───────────────────────────────────────────╮
│ Недостаточно данных – требуется дополнительная информация от других агентов.                     │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ На текущей доске собраны лишь методологические предложения, перечни требуемых исходных данных,   │
│ структура управления проектом и общие рекомендации по формированию стратегии. Конкретные         │
│ статистические, экономические, демографические и инфраструктурные показатели, а также результаты │
│ опросов заинтересованных сторон отсутствуют. Поэтому информации недостаточно для формирования    │
│ окончательного целевого плана развития Гатчины на 2026‑2035 годы. Необходимо запросить у коллег  │
│ реальные данные и результаты предварительных исследований. Процесс следует продолжить.           │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ недостаточно, данные, продолжить                                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── 📌 Критик [9455fe] ───────────────────────────────────────╮
│ На доске нет бесполезных записей; все текущие заметки полезны и дополняют друг друга.            │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ После тщательного анализа всех записей, размещённых на общей доске, я не нашёл ни одной, которая │
│ была бы полностью бесполезной или избыточной.                                                    │
│                                                                                                  │
│ **Почему каждая запись полезна:**                                                                │
│ 1. **f564a3 (План решения задачи)** – предоставляет общий методологический каркас, который       │
│ служит «скелетом» для всей дальнейшей работы. Без него трудно согласовать последовательность     │
│ действий.                                                                                        │
│ 2. **5651c8 (Расширенный перечень исходных данных и стейкхолдеров)** – детализирует, какие       │
│ именно данные и откуда необходимо собрать, а также расширяет список заинтересованных сторон. Это │
│ конкретизирует пункт 2 и 3 из общего плана.                                                      │
│ 3. **9b5312 (Экономический вклад)** – фокусируется на экономических показателях, их источниках и │
│ стратегических направлениях, что дополняет общие рекомендации и обеспечивает экономическую       │
│ основу для целей.                                                                                │
│ 4. **e2ff25 (Стратегический план‑дорожная карта)** – описывает управленческую структуру проекта, │
│ интегрированную базу данных и процесс согласования, тем самым превращая методологию в            │
│ практический механизм реализации.                                                                │
│ 5. **24d251 (Арбитр)** – указывает на текущий недостаток реальных данных и подчеркивает          │
│ необходимость их получения, что помогает всем участникам понять, какие шаги следует выполнить    │
│ дальше.                                                                                          │
│                                                                                                  │
│ Все записи охватывают разные уровни проекта (методология, данные, экономический анализ,          │
│ управленческая структура и текущий статус), поэтому ни одна из них не является дублирующей или   │
│ лишней.                                                                                          │
│                                                                                                  │
│ **Вывод:** бесполезных или избыточных записей на доске нет. Ожидаю дальнейших инструкций, новых  │
│ данных или новых записей для продолжения работы над целеполаганием г. Гатчина (2026‑2035 гг.).   │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ доска, записи, анализ, бесполезные, избыточные                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

### Summarize

In [10]:
summarizer_actor_agent = Agent(
    tools=board.tools,  # оставляем инструменты
    system_prompt="""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    
    Поставленная задача:
    `{question}`
    
    Алгоритм:
    1. Вызовите get_notes(), чтобы увидеть все записи
    2. Вызовите get_note() для КАЖДОЙ из них
    3. Синтезируйте финальный ответ
    4. Верните ТОЛЬКО финальный ответ, без лишних комментариев
    """.format(question=board.question)
)

summarizer_critic_agent = Agent(
    tools=board.tools,  # оставляем инструменты
    system_prompt="""
    Ваша задача: на основе черновика финального ответа на поставленную задачу и материалов на доске выдайте список замечаний.
    
    Поставленная задача:
    `{question}`
    """.format(question=board.question)
)

In [11]:
messages = []

n_gen = 3

for i in tqdm(range(n_gen)):
    actor_res = summarizer_actor_agent.invoke([HumanMessage(m) for m in messages[-1:]])
    messages.append(actor_res)

    if (i+1) < n_gen:
        critic_res = summarizer_critic_agent.invoke([HumanMessage(m) for m in messages[-1:]])
        messages.append(critic_res)

100%|██████████| 3/3 [10:32<00:00, 211.00s/it]


In [12]:
board.notes

[Note(content='План решения задачи:\n\n1. **Формулировка задачи** – собрать и систематизировать информацию о текущем состоянии города Гатчина, определить ключевые направления развития и сформировать цели на 2026‑2035 годы.\n\n2. **Сбор исходных данных**\n   - демографические, экономические, социальные и инфраструктурные показатели;\n   - существующие стратегии и программы развития (на уровне муниципалитета, региона, федеральные проекты);\n   - результаты предыдущих планов (2016‑2025) и их оценка.\n\n3. **Анализ заинтересованных сторон**\n   - органы местного самоуправления, бизнес‑сообщество, общественные организации, жители.\n   - проведение интервью/опросов, формирование требований и ожиданий.\n\n4. **SWOT‑анализ** города (сильные/слабые стороны, возможности, угрозы).\n\n5. **Определение стратегических направлений** (например, экономическое развитие, жилищно‑коммунальное хозяйство, транспорт, экология, культура, цифровизация).\n\n6. **Формулирование целей и задач** для каждого направ

In [13]:
messages

['**Целеполагание г.\u202fГатчина (Ленинградская область)\u202fна 2026‑2035\u202fгоды**  \n\n---\n\n## 1.\u202fОбщая методология (по\u202fплану\u202ff564a3)\n\n1. **Формулировка задачи** – определить текущее состояние города и сформулировать цели развития до\u202f2035\u202fгода.  \n2. **Сбор и систематизация исходных данных** (демография, экономика, инфраструктура, экология, цифровая среда, социальные услуги).  \n3. **Анализ заинтересованных сторон** (власть, бизнес, НКО, жители).  \n4. **SWOT‑анализ** города.  \n5. **Определение стратегических направлений**.  \n6. **Формулирование SMART‑целей** для каждого направления.  \n7. **Разработка мероприятий и проектов** (сроки, ответственные, ресурсы).  \n8. **Определение KPI и методики мониторинга**.  \n9. **Подготовка и публичное обсуждение стратегического документа**.  \n10. **Утверждение и план реализации** (календарный график, бюджет, распределение ответственности).  \n\n---\n\n## 2.\u202fРасширенный перечень исходных данных  \n\n| Катег

In [14]:
print(messages[-1])

**Город Гатчина (Ленинградская область)  
Стратегический план «Целеполагание 2026‑2035»**  

---

## 1. Исходные данные и базовый профиль

### 1.1 Демографический базис (2023 г.)  

| Показатель | Значение | Источник | Период обновления |
|------------|----------|----------|-------------------|
| Население, чел. | 93 800 | Росстат, муниципальная статистика | ежегодно |
| Доля населения ≥ 65 лет, % | 18,2 % | Росстат | ежегодно |
| Естественный прирост, % | –0,4 % | Росстат | ежегодно |
| Миграционный прирост, % | +0,7 % | Росстат | ежегодно |
| Средняя продолжительность жизни, лет | 78,5 | Росстат | ежегодно |

*Обновление: каждый год, публикация в открытом муниципальном портале.*

### 1.2 Экономический базис (2023 г.)  

| Показатель | Значение | Источник | Период |
|------------|----------|----------|--------|
| ВРП муниципального уровня, млрд руб. | 12,4 | Росстат, региональная служба эконом. развития | 2010‑2025 |
| Отраслевая структура (% от ВРП) | Промышленность 30, Строительство